# 3. Handle Encoding

This notebook converts the categorical features from the binned dataset into numeric columns for downstream modeling.

## Pipeline
1. Load the binned dataset
2. Select columns for one-hot and label encoding
3. Apply one-hot encoding to `merchant_category`
4. Apply ordered label encoding to `device_trust_score_binned`
5. Verify the encoded result
6. Save the encoded dataset for the next preprocessing step

## 1. Imports & Setup

In [24]:
import os
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Imports successful.')

Imports successful.


## 2. Load Binned Data

In [25]:
INPUT_PATH = 'C:/Users/2021ICTS28/Desktop/end-to-end-credit-card-fraud-detection-system/dataset/processed/credit_card_fraud_binned.csv'

df = pd.read_csv(INPUT_PATH)
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

Loaded: 10,000 rows x 10 columns


,transaction_id,amount,transaction_hour,merchant_category,foreign_transaction,location_mismatch,velocity_last_24h,cardholder_age,is_fraud,device_trust_score_binned
0,1,84.4700,22,Electronics,0,0,3,40,0,Good
1,2,529.8425,3,Travel,1,0,1,64,0,Excellent
2,3,237.0100,17,Grocery,0,0,1,61,0,Fair
3,4,164.3300,4,Grocery,0,1,3,34,0,Good
4,5,30.5300,15,Food,0,0,0,44,0,Excellent


## 3. Select Columns to Encode

In [26]:
TARGET_COL = 'is_fraud'
ID_COL = 'transaction_id' if 'transaction_id' in df.columns else None

one_hot_cols = [col for col in ['merchant_category'] if col in df.columns]
label_col = 'device_trust_score_binned' if 'device_trust_score_binned' in df.columns else None
label_mapping = {'Poor': 0, 'Fair': 1, 'Good': 2, 'Excellent': 3}

if not one_hot_cols and label_col is None:
    raise ValueError('No columns found to encode.')

print('Columns selected for one-hot encoding:', one_hot_cols)
print('Label-encoded column:', label_col)
print('Label order:', label_mapping)

Columns selected for one-hot encoding: ['merchant_category']
Label-encoded column: device_trust_score_binned
Label order: {'Poor': 0, 'Fair': 1, 'Good': 2, 'Excellent': 3}


## 4. Apply Encoding

In [27]:
df_encoded = pd.get_dummies(df, columns=one_hot_cols, drop_first=False, dtype=int)

if label_col is not None:
    unknown_labels = sorted(set(df_encoded[label_col].dropna().unique()) - set(label_mapping.keys()))
    if unknown_labels:
        raise ValueError(f'Unexpected values in {label_col}: {unknown_labels}')
    df_encoded[label_col] = df_encoded[label_col].map(label_mapping).astype('Int64')

print(f'One-hot encoded {len(one_hot_cols)} column(s) and label encoded {1 if label_col is not None else 0} column(s).')
df_encoded.head()

One-hot encoded 1 column(s) and label encoded 1 column(s).


,transaction_id,amount,transaction_hour,foreign_transaction,location_mismatch,velocity_last_24h,cardholder_age,is_fraud,device_trust_score_binned,merchant_category_Clothing,merchant_category_Electronics,merchant_category_Food,merchant_category_Grocery,merchant_category_Travel
0,1,84.4700,22,0,0,3,40,0,2,0,1,0,0,0
1,2,529.8425,3,1,0,1,64,0,3,0,0,0,0,1
2,3,237.0100,17,0,0,1,61,0,1,0,0,0,1,0
3,4,164.3300,4,0,1,3,34,0,2,0,0,0,1,0
4,5,30.5300,15,0,0,0,44,0,3,0,0,1,0,0


## 5. Verify the Transformation

In [28]:
print('Dtype summary after encoding:')
print(df_encoded.dtypes.value_counts())

one_hot_encoded_cols = [col for col in df_encoded.columns if any(col.startswith(prefix + '_') for prefix in one_hot_cols)]
print()
print('One-hot encoded columns created:')
print(one_hot_encoded_cols)

print()
print('Label-encoded column sample:')
if label_col is not None:
    print(df_encoded[[label_col]].head())

Dtype summary after encoding:
int64      12
float64     1
Int64       1
Name: count, dtype: int64

One-hot encoded columns created:
['merchant_category_Clothing', 'merchant_category_Electronics', 'merchant_category_Food', 'merchant_category_Grocery', 'merchant_category_Travel']

Label-encoded column sample:
   device_trust_score_binned
0                          2
1                          3
2                          1
3                          2
4                          3


## 6. Save Outputs

In [29]:
OUTPUT_PATH = 'C:/Users/2021ICTS28/Desktop/end-to-end-credit-card-fraud-detection-system/dataset/processed/credit_card_fraud_encoded.csv'

output_dir = os.path.dirname(OUTPUT_PATH)
os.makedirs(output_dir, exist_ok=True)
df_encoded.to_csv(OUTPUT_PATH, index=False)

print(f'Saved encoded dataset to: {OUTPUT_PATH}')
print(f'Rows: {df_encoded.shape[0]:,} | Columns: {df_encoded.shape[1]:,}')

Saved encoded dataset to: C:/Users/2021ICTS28/Desktop/end-to-end-credit-card-fraud-detection-system/dataset/processed/credit_card_fraud_encoded.csv
Rows: 10,000 | Columns: 14
